In [ ]:
# Cell 1 - Cài đặt thư viện
!pip install -q --upgrade transformers trl==0.7.0
!pip install -q accelerate bitsandbytes peft datasets
!pip install -q sentencepiece

print("✅ Đã cài đặt xong các thư viện")

In [ ]:
# Cell 2 - Tải dữ liệu quiz
from datasets import load_dataset

# 🔹 Tải dữ liệu quiz từ file jsonl
data = load_dataset("json", data_files="/kaggle/input/quizdata/10kQuiz.jsonl")

# 🔹 Kiểm tra dữ liệu
print("📊 Thông tin dataset:")
print(data)

# 🔹 Xem mẫu dữ liệu đầu tiên
print("\n📝 Mẫu dữ liệu đầu tiên:")
print(data["train"][0])

In [ ]:
# Cell 3 - Định dạng dữ liệu theo template Qwen
from transformers import AutoTokenizer

model_id = "Qwen/Qwen3-4B"  # Sửa tên model cho chính xác

# 🔹 Tải tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 🔹 Hàm định dạng dữ liệu
def format_chat(example):
    """
    Chuyển danh sách messages trong mỗi mẫu dữ liệu 
    thành text hoàn chỉnh theo template chat của Qwen.
    """
    text = tokenizer.apply_chat_template(
        example["messages"], 
        tokenize=False, 
        add_generation_prompt=False
    )
    return {"text": text}

# 🔹 Map dữ liệu qua hàm format_chat
train_dataset = data["train"].map(format_chat)

# ✅ Kiểm tra kết quả
print("✅ Đã định dạng dữ liệu:")
print(train_dataset[0]["text"][:500] + "...")  # In 500 ký tự đầu

In [ ]:
# Cell 4 - Fine-tuning với LoRA (SỬA LẠI THEO FILE THÀNH CÔNG)
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
import torch
import os

# Suppress specific warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# --- Configurable ---
model_id = "Qwen/Qwen3-4B"
output_dir = "/kaggle/working/qwen_quiz_finetuned"
max_length = 512
batch_size = 1
grad_accum = 4
num_epochs = 1
learning_rate = 2e-4

print("🚀 Đang khởi tạo fine-tuning Qwen3-4B...")

# --- Load tokenizer & model ---
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load model với cấu hình tối ưu
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("✅ Đã tải model và tokenizer")

# --- Tokenize dataset ---
def tokenize_fn(batch):
    out = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=max_length)
    out["labels"] = out["input_ids"].copy()
    return out

tokenized = train_dataset.map(tokenize_fn, batched=True, remove_columns=train_dataset.column_names)
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"✅ Đã tokenize {len(tokenized)} samples")

# --- Data collator ---
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# --- PEFT (LoRA) config ---
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap model với LoRA
model = get_peft_model(model, peft_config)
print("✅ Đã áp dụng LoRA")

# --- TrainingArguments ---
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    num_train_epochs=num_epochs,
    learning_rate=learning_rate,
    fp16=True,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    remove_unused_columns=False,
    report_to=[],  # Tắt logging để giảm warnings
)

# --- Trainer ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("🎯 Bắt đầu training...")

# --- Train ---
try:
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print("✅ Fine-tune thành công! Model đã lưu tại:", output_dir)
except RuntimeError as e:
    print("❌ Lỗi trong quá trình training:", str(e))
    if "out of memory" in str(e).lower():
        print("💡 Gợi ý: Giảm max_length xuống 256 hoặc batch_size")
except Exception as e:
    print("❌ Lỗi không xác định:", str(e))

In [ ]:
# Cell 5 - Lưu model
trainer.save_model("/kaggle/working/qwen_quiz_finetuned")
tokenizer.save_pretrained("/kaggle/working/qwen_quiz_finetuned")

print("✅ Đã lưu model và tokenizer")

In [2]:
# Cell 6 - Test model sau khi fine-tune
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Tải model đã fine-tune
model_path = "/kaggle/input/qwen-genquiz-finetuned/transformers/default/1/qwen_quiz_finetuned"

print("Đang tải model đã fine-tune...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model đã sẵn sàng để test!")

Đang tải model đã fine-tune...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model đã sẵn sàng để test!


In [3]:
# Cell 7 - Mô phỏng hệ thống tự động tạo quiz với context RAG mẫu

import random
import torch

# Context RAG mẫu từ database
RAG_CONTEXT_EXAMPLE = {
    "text": "Trong 15 năm đầu của sự nghiệp đổi mới (1986-2000), Việt Nam đã đạt được những thành tựu đáng tự hào, được cộng đồng quốc tế cổ vũ và xem là bài học kinh nghiệm cho sự nghiệp phát triển của mình. Quá trình chuyển đổi sang nền kinh tế thị trường ở Việt Nam đã không gây xáo trộn xã hội hay đổ vỡ chính trị như các nước XHCN Đông Âu và Liên Xô trước đây. Việt Nam đã giảm nhanh tình trạng nghèo đói, bước đầu xây dựng nền kinh tế công nghiệp hóa, đạt tốc độ tăng trưởng kinh tế cao đi đôi với sự công bằng tương đối trong xã hội. Đến năm 2000, Việt Nam đã trở thành một nước có tốc độ tăng trưởng cao trong khu vực và từng bước xác lập vai trò, vị thế trong hội nhập khu vực và quốc tế.",
    "metadata": {
        "nam": None,
        "trieu_dai": "Thời kỳ Đổi mới", 
        "thuc_the": ["Việt Nam", "Đông Âu", "Liên Xô"],
        "chu_de": "Thành tựu đổi mới và hội nhập quốc tế"
    }
}

def create_quiz_from_rag_context():
    """
    Hệ thống tự động tạo quiz từ context RAG khi người dùng bấm nút
    """
    # 1. Lấy context từ RAG (trong thực tế sẽ query từ vector DB)
    context_text = RAG_CONTEXT_EXAMPLE["text"]
    
    # 2. Tự động tạo prompt với context ẩn
    prompt = f"""Hãy tạo một câu hỏi trắc nghiệm với 4 lựa chọn (3 sai, 1 đúng) và giải thích đáp án dựa trên thông tin sau:

{context_text}

Yêu cầu:
1. Tạo câu hỏi trắc nghiệm với 4 lựa chọn (A, B, C, D)
2. Chỉ có 1 đáp án đúng, 3 đáp án sai phải có vẻ hợp lý
3. Giải thích ngắn gọn tại sao đáp án đó đúng"""
    
    # 3. Tạo messages cho model
    messages = [
        {"role": "system", "content": "Bạn là chuyên gia lịch sử kinh tế Việt Nam. Hãy tạo câu hỏi trắc nghiệm dựa trên ngữ liệu được cung cấp."},
        {"role": "user", "content": prompt}
    ]
    
    # 4. Generate câu hỏi từ model
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    input_len = inputs.input_ids.shape[-1]
    
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )
    
    new_tokens = outputs[0, input_len:]
    quiz = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    return {
        "context_topic": RAG_CONTEXT_EXAMPLE["metadata"]["chu_de"],
        "context_period": RAG_CONTEXT_EXAMPLE["metadata"]["trieu_dai"],
        "generated_quiz": quiz
    }

# Test hệ thống
print("🚀 HỆ THỐNG TẠO QUIZ TỰ ĐỘNG VỚI CONTEXT RAG")
print("=" * 70)
print("Mô phỏng: Người dùng bấm nút 'Tạo Câu Hỏi'")
print("Hệ thống tự động:")
print("1. Lấy context từ RAG database")
print("2. Tạo prompt với context ẩn")
print("3. Generate câu hỏi trắc nghiệm")
print("=" * 70)

print("\n📊 THÔNG TIN CONTEXT RAG:")
print(f"Chủ đề: {RAG_CONTEXT_EXAMPLE['metadata']['chu_de']}")
print(f"Thời kỳ: {RAG_CONTEXT_EXAMPLE['metadata']['trieu_dai']}")
print(f"Thực thể: {', '.join(RAG_CONTEXT_EXAMPLE['metadata']['thuc_the'])}")
print(f"\n📝 Nội dung context (ẩn với người dùng):")
print(RAG_CONTEXT_EXAMPLE['text'][:150] + "...")
print("\n" + "=" * 70)

# Mô phỏng người dùng bấm nút
print("\n🎯 KẾT QUẢ QUIZ ĐƯỢC TẠO:")
print("-" * 70)

result = create_quiz_from_rag_context()

print(f"📌 Chủ đề: {result['context_topic']}")
print(f"⏳ Thời kỳ: {result['context_period']}")
print(f"\n📖 CÂU HỎI TRẮC NGHIỆM:")
print("-" * 40)
print(result['generated_quiz'])
print("-" * 40)

# Kiểm tra chất lượng quiz
print("\n✅ KIỂM TRA CHẤT LƯỢNG QUIZ:")
print("-" * 40)

quiz_text = result['generated_quiz']

# Kiểm tra các thành phần
has_question = "Câu hỏi:" in quiz_text or "?" in quiz_text
has_options = any(x in quiz_text for x in ["A.", "B.", "C.", "D."])
has_correct_answer = "Đáp án đúng:" in quiz_text or "Đáp án:" in quiz_text
has_explanation = "Giải thích:" in quiz_text or "Giải thích" in quiz_text

print(f"✓ Có câu hỏi: {'CÓ' if has_question else 'KHÔNG'}")
print(f"✓ Có 4 đáp án: {'CÓ' if has_options else 'KHÔNG'}")
print(f"✓ Có đáp án đúng: {'CÓ' if has_correct_answer else 'KHÔNG'}")
print(f"✓ Có giải thích: {'CÓ' if has_explanation else 'KHÔNG'}")

# Tạo thêm 1 quiz nữa để so sánh
print("\n\n🔄 TEST LẦN 2 - CÙNG CONTEXT:")
print("-" * 70)
result2 = create_quiz_from_rag_context()
print(result2['generated_quiz'][:200] + "...")
print("-" * 40)

print("\n✅ HOÀN THÀNH TEST!")
print("💡 Trong ứng dụng thực tế:")
print("   - Context sẽ được lấy ngẫu nhiên từ vector database")
print("   - Mỗi lần bấm nút sẽ tạo câu hỏi khác nhau")
print("   - Người dùng không bao giờ thấy context gốc")

🚀 HỆ THỐNG TẠO QUIZ TỰ ĐỘNG VỚI CONTEXT RAG
Mô phỏng: Người dùng bấm nút 'Tạo Câu Hỏi'
Hệ thống tự động:
1. Lấy context từ RAG database
2. Tạo prompt với context ẩn
3. Generate câu hỏi trắc nghiệm

📊 THÔNG TIN CONTEXT RAG:
Chủ đề: Thành tựu đổi mới và hội nhập quốc tế
Thời kỳ: Thời kỳ Đổi mới
Thực thể: Việt Nam, Đông Âu, Liên Xô

📝 Nội dung context (ẩn với người dùng):
Trong 15 năm đầu của sự nghiệp đổi mới (1986-2000), Việt Nam đã đạt được những thành tựu đáng tự hào, được cộng đồng quốc tế cổ vũ và xem là bài học k...


🎯 KẾT QUẢ QUIZ ĐƯỢC TẠO:
----------------------------------------------------------------------
📌 Chủ đề: Thành tựu đổi mới và hội nhập quốc tế
⏳ Thời kỳ: Thời kỳ Đổi mới

📖 CÂU HỎI TRẮC NGHIỆM:
----------------------------------------
<think>

</think>

**Câu hỏi:** Trong giai đoạn 1986-2000, Việt Nam đã ghi nhận những kết quả nổi bật nào về mặt kinh tế?

**Đáp án:**
A. Nền kinh tế còn quá phụ thuộc vào xuất khẩu nông sản.
B. Tình trạng nghèo đói gia tăng nghiêm trọn